<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: An IVR for spoken intents

**[An IVR for spoken intents](https://learning.nextia-ai.com/courses/speech/m10/an-ivr-for-spoken-intents/)** · Speech AI: Transcribe, Understand and Speak · Case study 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** run the intent step of an IVR on real spoken banking requests in four language varieties: keyword rules, a TF-IDF classifier on Whisper transcripts with cross-validation, and a recorded LLM. You will measure accuracy, the share of calls routed right without a person, and the cost of 1,000 calls, by speaker group, then choose a design and run it on new input.

| | |
|---|---|
| **Time** | About 2.5 hours with the lesson page; the notebook runs in a few minutes |
| **Needs** | An internet connection for the setup cells (about 3 MB of data and the 480 MB Whisper small model). No account and no key: the LLM's answers are recorded. |
| **You should know** | Keyword rules and the classifier from [Why did they call?](https://learning.nextia-ai.com/courses/speech/m06/why-did-they-call/); the IVR's intent step from [Let callers say what they need](https://learning.nextia-ai.com/courses/speech/m07/let-callers-say-what-they-need/); containment from [Test the IVR like a product](https://learning.nextia-ai.com/courses/speech/m07/test-the-ivr-like-a-product/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0, matplotlib 3.11.2, scikit-learn 1.9.1 and faster-whisper 1.2.1, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m10/an-ivr-for-spoken-intents/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M10-L04-an-ivr-for-spoken-intents/an-ivr-for-spoken-intents-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, scikit-learn 1.9.1, faster-whisper 1.2.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. faster-whisper brings its own engine (CTranslate2) and the Silero voice detector: the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2 scikit-learn==1.9.1 faster-whisper==1.2.1
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile', 'scikit-learn', 'faster-whisper']:
    print(name, version(name))

## Setup: get the requests

Run the next cell. It downloads the files of this case study (about 3 MB) into a folder `minds-ivr`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `minds-ivr`, then run the cell again.

What you get: the text and labels of all 2,348 requests (US, British and Australian English, and French), the **recorded** Whisper base and Whisper small transcripts of every request, the **recorded** answers of an LLM, and 12 short recordings for the live run at the end.

Source: [MInDS-14](https://huggingface.co/datasets/PolyAI/minds14), PolyAI (Gerz et al., 2021), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Nextia Learning converted the 12 recordings from mu-law to 16-bit WAV; the transcripts and intents are the dataset's own; the Whisper and LLM outputs are ours. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M10-L04-an-ivr-for-spoken-intents/dataset.md) has the details.

In [ ]:
# Setup: download the files into minds-ivr/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
HERE_FOLDER = "M10-L04-an-ivr-for-spoken-intents/"
FILES = {
    "data/audio/en-AU-136.wav": "529c00c676d5d1634f4f5c2fa6a09434293ac00a0f87ce9bc275b2a4ee88ef19",
    "data/audio/en-AU-178.wav": "9623a0b9a1ab3afc42822f1175b860c972452c32a0e37801ab63172cc0e39a48",
    "data/audio/en-AU-553.wav": "c01374a4fc657f1e2624a457b6c25616254615b9b2f3845460b6abf385abc909",
    "data/audio/en-GB-259.wav": "d6ab7f9cae0164b26d5cd28f79577ccaa3e21c18773a735e9b61584b4c1c96f2",
    "data/audio/en-GB-316.wav": "5a4ed8f7646b831a35b1d30494ecaf62dad502333f529ae2682c9d242f10f181",
    "data/audio/en-GB-363.wav": "bfe00946a52dbb6b6b6897aca8689d904c674b16d719074f61001740cd6c92f7",
    "data/audio/en-US-081.wav": "6a18e9285da091533ff640aaeb16b0c7a2163c34a83bf552b81e2f9e07db1e67",
    "data/audio/en-US-151.wav": "ec7d89323e0180616084898fbe733ee27b3f880829a4cab65d2937841b34d2a0",
    "data/audio/en-US-347.wav": "75c8cb0e9375ff776bcc09e43fc428b4a80ff6ebb8adf1569a8be58de5f69847",
    "data/audio/fr-FR-219.wav": "1798cc1df247d2b216adc88873f5b914926c4e31711ef46bbf0b6a9a63d6c0a7",
    "data/audio/fr-FR-256.wav": "aa340c746dcd96113253f8caaa985bf00c93774692dca03b82e10146d350759d",
    "data/audio/fr-FR-473.wav": "550dd7038c3643b2f41281d2c21d4bd1ede6bee3d5f256de47d60c951a15a1b2",
    "data/clips.jsonl": "ff09063738e0ffb165c75fe5c6c09aeb01c89a0265bd8fdc2522116d566b6e42",
    "data/recorded/llm-dataset.jsonl": "04719af0ecc374dbb78a761674c4203f2cb151cad6cbb0bddc6a0719e03bb8ba",
    "data/recorded/llm-whisper-base-v2.jsonl": "e8cc292f17f6d4a3bec2f6added777c5300c93057db37a2faff6b076096c509c",
    "data/recorded/llm-whisper-base.jsonl": "522f377be3563905b16ee4b69ddb962bff026c8744fe000defec5e5c8c68a16c",
    "data/recorded/whisper-base.jsonl": "8314870099655e4e67462fee53479e65c85e118ba0c7ea66dd7c9a766dc0ca30",
    "data/recorded/whisper-small.jsonl": "5091536d8a53e03fa27a383489de783cd1959940ca79ce163fc68661c969c9a4",
}
FILES["compare.py"] = "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b"
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("minds-ivr").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + (name if name == "compare.py" else HERE_FOLDER + name), path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder minds-ivr and run again.")
os.chdir(PROJECT)
print(f"Ready: {len(FILES)} files in minds-ivr.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

## Setup: the intent tools

The next cell defines the tools of this case study. They are the course's tools, made ready for 14 banking intents:

- `find_intent(text, language)`: **keyword rules** in the shape of the Larkfield IVR's rules (from [Let callers say what they need](https://learning.nextia-ai.com/courses/speech/m07/let-callers-say-what-they-need/)): a list of words for each intent, and the last intent the caller names wins. `None` means no match.
- `IntentClassifier`: the **TF-IDF and logistic regression** classifier of [Why did they call?](https://learning.nextia-ai.com/courses/speech/m06/why-did-they-call/), now with a probability for each intent.
- `cross_val(texts, labels, strata)`: **cross-validation**. Each request is scored by a model that never saw it.
- `route(...)`, `tally(...)` and `outcome_cost(...)`: what happens to a call (routed right, routed wrong, a person) and what that costs for 1,000 calls.

Run the cell. It prints the 14 intents.

In [ ]:
import numpy as np

normalise = cp.normalise

INTENTS = ["abroad", "address", "app_error", "atm_limit", "balance", "business_loan", "card_issues", "cash_deposit",
           "direct_debit", "freeze", "high_value_payment", "joint_account", "latest_transactions", "pay_bill"]

# Written once from the intent names and their usual words, in English and French, before looking at any result.
INTENT_WORDS = {
    "abroad": ["abroad", "overseas", "travel", "travelling", "traveling", "trip", "holiday", "vacation", "foreign",
               "another country", "currency", "étranger", "voyage", "voyager", "vacances", "devise", "devises"],
    "address": ["address", "moved", "moving", "move house", "adresse", "déménagé", "déménager", "déménagement"],
    "app_error": ["app", "application", "error", "not working", "doesn't work", "does not work", "crash", "log in",
                  "login", "appli", "erreur", "bug", "fonctionne pas", "marche pas", "connecter"],
    "atm_limit": ["atm", "cash machine", "cashpoint", "withdraw", "withdrawal", "limit", "distributeur", "retrait",
                  "retirer", "plafond", "limite"],
    "balance": ["balance", "how much money", "how much is in", "how much do i have", "solde", "combien"],
    "business_loan": ["business", "loan", "company", "entreprise", "prêt", "crédit", "société", "professionnel"],
    "card_issues": ["lost my card", "lost card", "stolen", "card is blocked", "card has been", "card was", "card is not",
                    "card doesn't", "card isn't", "new card", "declined", "damaged", "carte perdue", "perdu ma carte",
                    "volé", "volée", "carte bloquée", "nouvelle carte", "ma carte ne", "carte est", "carte a"],
    "cash_deposit": ["deposit", "pay in", "paying in", "cash in", "put cash", "put money", "dépôt", "déposer",
                     "verser", "espèces", "liquide"],
    "direct_debit": ["direct debit", "standing order", "debit", "prélèvement", "prélèvements", "domiciliation"],
    "freeze": ["freeze", "frozen", "suspend", "lock my account", "block my account", "geler", "gel", "bloquer mon compte",
               "suspendre", "bloquer"],
    "high_value_payment": ["large payment", "big payment", "large amount", "large sum", "high value", "large transfer",
                           "big transfer", "transfer", "thousand", "gros montant", "grosse somme", "somme importante",
                           "montant élevé", "virement", "mille"],
    "joint_account": ["joint", "shared account", "partner", "wife", "husband", "compte joint", "compte commun",
                      "conjoint", "conjointe", "mari", "femme"],
    "latest_transactions": ["transactions", "transaction", "recent", "statement", "last payments", "history",
                            "spending", "opérations", "relevé", "historique", "dernières", "derniers"],
    "pay_bill": ["bill", "pay a bill", "invoice", "facture", "factures", "payer"],
}


def find_intent(text, language):
    """Keyword rules. The LAST intent the caller names wins; None when no word matches."""
    t = " " + normalise(text, "basic", language) + " "
    best, pos = None, -1
    for intent, words in INTENT_WORDS.items():
        for w in words:
            i = t.rfind(" " + normalise(w, "basic", language) + " ")
            if i > pos:
                best, pos = intent, i
    return best


class IntentClassifier:
    """TF-IDF of words and word pairs + logistic regression (the project's ReasonClassifier, with probabilities)."""

    def fit(self, texts, labels):
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.linear_model import LogisticRegression
        self.vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
        self.lr = LogisticRegression(max_iter=2000, C=10.0)
        self.lr.fit(self.vec.fit_transform([normalise(t, "basic") for t in texts]), labels)
        return self

    def proba(self, texts):
        """(classes, probabilities): one row per text."""
        return list(self.lr.classes_), self.lr.predict_proba(self.vec.transform([normalise(t, "basic") for t in texts]))


def cross_val(texts, labels, strata, folds=5, seed=0):
    """Out-of-fold probabilities: each clip is scored by a model that never saw it (stratified by `strata`)."""
    from sklearn.model_selection import StratifiedKFold
    texts, labels = list(texts), list(labels)
    out = np.zeros((len(texts), len(INTENTS)))
    for tr, te in StratifiedKFold(folds, shuffle=True, random_state=seed).split(texts, strata):
        cls, p = IntentClassifier().fit([texts[i] for i in tr], [labels[i] for i in tr]).proba([texts[i] for i in te])
        for j, c in enumerate(cls):
            out[te, INTENTS.index(c)] = p[:, j]
    return out


def route(top, sure, truth, shortlist=(), menu_ok=0.9):
    """What happens to one call, as shares that add up to 1: {"right", "wrong", "person", "menu"}.
    top: the method's intent (None = no answer); sure: sure enough to route; shortlist: the intents a short keypad
    menu offers when it is not sure (empty: straight to a person). Assumption (labelled on the page): when the right
    intent is on the short menu, 90% of callers press its key (menu_ok); the others press 0 or give up: a person.
    "menu" is the share of callers who heard the extra menu (not a cost by itself)."""
    if top is not None and sure:
        return {"right": float(top == truth), "wrong": float(top != truth), "person": 0.0, "menu": 0.0}
    if shortlist:
        ok = menu_ok if truth in shortlist else 0.0
        return {"right": ok, "wrong": 0.0, "person": 1 - ok, "menu": 1.0}
    return {"right": 0.0, "wrong": 0.0, "person": 1.0, "menu": 0.0}


def tally(outcomes):
    """Mean shares over many calls."""
    n = max(len(outcomes), 1)
    return {k: sum(o[k] for o in outcomes) / n for k in ("right", "wrong", "person", "menu")}


def outcome_cost(shares, person=4.00, wrong=15.00, per_call=0.0):
    """US$ per 1,000 calls: a person US$4.00 a call, a wrong route US$15 (the course's assumptions), compute per call."""
    return 1000 * (shares["person"] * person + shares["wrong"] * wrong + per_call)

print(len(INTENTS), 'intents:', ', '.join(INTENTS))

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data, briefly

MInDS-14 has spoken requests to a bank's phone line, each labelled with one of 14 intents. We use four language varieties: US, British and Australian English, and French from France. Each request is a short phone recording (8 kHz).

Run the next cell. It prints the size of each variety and three requests.

In [ ]:
from collections import Counter
clips = [json.loads(line) for line in open("data/clips.jsonl", encoding="utf-8")]
by_id = {c["id"]: c for c in clips}
VARIETIES = ["en-US", "en-GB", "en-AU", "fr-FR"]
for v in VARIETIES:
    mine = [c for c in clips if c["variety"] == v]
    print(f"{v}: {len(mine)} requests, {sum(c['seconds'] for c in mine) / 3600:.2f} hours, "
          f"median {np.median([c['seconds'] for c in mine]):.1f} s")
print("\nThree requests:")
for cid in ["en-US-151", "en-GB-316", "fr-FR-256"]:
    c = by_id[cid]
    print(f"{cid} [{c['intent']}] {c['transcript']!r}" + (f" (English: {c['english']!r})" if c["language"] == "fr" else ""))

> **Note.** The `transcript` field was **not** written by a person. The dataset's authors made it with Google's speech recogniser. The **intent** is the truth here; every transcript is a recogniser's guess.

Listen to one request. Then compare the two machine transcripts.

In [ ]:
display(Audio("data/audio/en-GB-316.wav"))
print("Dataset text (Google's recogniser):", by_id["en-GB-316"]["transcript"])
base_text = {r["id"]: r["text"] for r in map(json.loads, open("data/recorded/whisper-base.jsonl", encoding="utf-8"))}
print("Whisper base (recorded):          ", base_text["en-GB-316"])

## 2. Two recognisers, and how often they disagree

We ran Whisper base (the Larkfield IVR's recogniser) and Whisper small on every request and recorded the text. Without a human reference, we cannot measure a word error rate. We can measure how often two recognisers **disagree**: the same calculation as the WER of [Word error rate, step by step](https://learning.nextia-ai.com/courses/speech/m02/word-error-rate-step-by-step/), with the dataset's text in the place of the reference.

> **Predict.** Which variety will have the most disagreement?

In [ ]:
def read_jsonl(path):
    return {r["id"]: r for r in map(json.loads, open(path, encoding="utf-8"))}
whisper = {size: read_jsonl(f"data/recorded/whisper-{size}.jsonl") for size in ("base", "small")}
TEXT = {"dataset": {c["id"]: c["transcript"] for c in clips},
        **{size: {k: r["text"] for k, r in rows.items()} for size, rows in whisper.items()}}

for size in ("base", "small"):
    for v in VARIETIES:
        mine = [c for c in clips if c["variety"] == v]
        s = cp.corpus_scores([(c["transcript"], TEXT[size][c["id"]], c["language"]) for c in mine], "full")
        rtf = sum(whisper[size][c["id"]]["seconds"] for c in mine) / sum(c["seconds"] for c in mine)
        print(f"Whisper {size:5s} {v}: disagrees with the dataset's text on {s['wer']:.1%} of the words; real-time factor {rtf:.3f}")

> **Check.** French has the most disagreement. A disagreement is an error of at least one of the two recognisers, but we cannot say which one.

## 3. A fair test: keep new input apart, then cross-validate

We keep 20% of each variety apart as **new input** for the end-to-end run. We never use it to choose anything. The other 80% is for the comparison. The baseline always answers the most common intent.

In [ ]:
from sklearn.model_selection import train_test_split
new = set()
for v in VARIETIES:
    ids = [c["id"] for c in clips if c["variety"] == v]
    _, kept_apart = train_test_split(ids, test_size=0.2, random_state=0, stratify=[by_id[i]["intent"] for i in ids])
    new |= set(kept_apart)
comp = [c["id"] for c in clips if c["id"] not in new]
new_ids = [c["id"] for c in clips if c["id"] in new]
print(f"{len(comp)} requests to compare the methods, {len(new_ids)} kept apart as new input")

def accuracy(pred, ids):
    """The share of requests whose predicted intent is the labelled one."""
    return sum(pred[i] == by_id[i]["intent"] for i in ids) / len(ids)

def by_variety(pred, ids):
    return {v: round(accuracy(pred, [i for i in ids if by_id[i]["variety"] == v]), 3) for v in VARIETIES}

most_common = Counter(by_id[i]["intent"] for i in comp).most_common(1)[0][0]
baseline = {i: most_common for i in comp}
print("Baseline, always", repr(most_common), "->", f"{accuracy(baseline, comp):.1%}")

> **Check.** 1877 requests to compare, 471 kept apart. The baseline is right 7.6% of the time: about 1 in 14.

## 4. Method 1: keyword rules

In [ ]:
rules = {i: find_intent(TEXT["base"][i], by_id[i]["language"]) for i in comp}
print(f"Keyword rules on Whisper base: {accuracy(rules, comp):.1%}", by_variety(rules, comp))
for v in VARIETIES:
    mine = [i for i in comp if by_id[i]["variety"] == v]
    print(f"  {v}: no word matched in {sum(rules[i] is None for i in mine) / len(mine):.0%} of the requests")

> **Check.** 67.1% overall, and only 49.4% in French, where no word matched in 39% of the requests. The rules were written once from the intent names, in the shape of the Larkfield IVR's rules, and never tuned.

## 5. Method 2: a classifier, with cross-validation

The sets are small (about 450 requests per variety), so we use **5-fold cross-validation**: the requests are cut into 5 parts; a model learns from 4 parts and scores the fifth; this repeats 5 times. Every request is scored once, by a model that never saw it.

In [ ]:
def oof(source, ids):
    """Out-of-fold probabilities: one English model (three varieties together) and one French model."""
    P = {}
    for lang in ("en", "fr"):
        sel = [i for i in ids if by_id[i]["language"] == lang]
        p = cross_val([TEXT[source][i] for i in sel], [by_id[i]["intent"] for i in sel],
                      [by_id[i]["variety"] + by_id[i]["intent"] for i in sel])
        P.update({i: p[k] for k, i in enumerate(sel)})
    return P

P = oof("base", comp)
clf = {i: INTENTS[int(np.argmax(P[i]))] for i in comp}
top3 = {i: [INTENTS[k] for k in np.argsort(-P[i])[:3]] for i in comp}
print(f"Classifier on Whisper base: {accuracy(clf, comp):.1%}", by_variety(clf, comp))
print(f"The right intent is in the top 3: {sum(by_id[i]['intent'] in top3[i] for i in comp) / len(comp):.1%}")

> **Check.** 95.0% overall (English 97.1%, French 88.2%). The right intent is in the top 3 for 99.1% of the requests: that is what makes a short menu useful.

## 6. Method 3: an LLM (recorded answers)

Azure chat-small read each Whisper base transcript (text only, never audio) and chose an intent, or "unclear", with up to two alternatives, from a list with one line about each intent. Its answers were recorded once, with two versions of the list. The second list rewrote five lines (for example "freeze: freezing, blocking or stopping a card or the account") after we read the first list's mistakes.

> **Predict.** How much can five lines of a prompt change the accuracy?

In [ ]:
def llm_intents(name):
    rec = read_jsonl(f"data/recorded/{name}.jsonl")
    return rec, {i: (rec[i]["answer"].get("intent") if rec[i]["answer"].get("intent") in INTENTS else None) for i in comp}

for name, label in (("llm-whisper-base", "first list"), ("llm-whisper-base-v2", "second list")):
    rec, llm = llm_intents(name)
    print(f"LLM ({label}, recorded) on Whisper base: {accuracy(llm, comp):.1%}", by_variety(llm, comp))
    print(f"  'unclear' for {sum(llm[i] is None for i in comp) / len(comp):.1%};",
          f"US${np.mean([r['usd'] for r in rec.values()]) * 1000:.3f} per 1,000 requests;",
          f"median {np.median([r['latency_s'] for r in rec.values()]):.2f} s an answer")
    mistakes = Counter((by_id[i]["intent"], llm[i] or "unclear") for i in comp if llm[i] != by_id[i]["intent"])
    print("  most common mistakes:", mistakes.most_common(3))

> **Check.** First list: 85.3% (most "freeze" requests went to "card_issues": our words, not the labels). Second list: 94.2%, French 90.5%, which is better than the classifier in French. It still learned nothing from the labelled requests: it only reads our descriptions.

## 7. From accuracy to routing and cost

An IVR does not only guess. It can **route** the call, **ask again** with a short keypad menu ("press 1 for …, 2 for …, 3 for …, 0 for a person"), or **hand over** to a person. The costs are the course's assumptions: a person US$4.00 a call, a wrong route US$15. One more assumption, of this case study: when the right intent is on the short menu, 9 callers in 10 press its key, and the others need a person.

> **Predict.** With a menu as the safety net, should the threshold be higher or lower than without it?

In [ ]:
def shares(ids, threshold, menu, P=P, top3=top3):
    """The classifier routes a call when its top probability is at least `threshold`; otherwise a person,
    or (menu=True) a short keypad menu with its top 3 intents."""
    return tally([route(top3[i][0], P[i].max() >= threshold, by_id[i]["intent"], top3[i] if menu else ())
                  for i in ids])

PER_CALL = 0.002 + 0.00002          # IVR compute and Whisper base on a CPU server (assumptions)
PER_CALL_SMALL = 0.002 + 0.00005    # the same with Whisper small
thresholds = [round(t, 2) for t in np.arange(0.1, 0.95, 0.05)]
cost = {menu: [outcome_cost(shares(comp, t, menu), per_call=PER_CALL) for t in thresholds] for menu in (False, True)}
for menu in (False, True):
    best = thresholds[int(np.argmin(cost[menu]))]
    s = shares(comp, best, menu)
    print(f"{'else a 3-item menu' if menu else 'else a person    '}: best threshold {best:.2f}, "
          f"routed right {s['right']:.1%}, wrong {s['wrong']:.1%}, person {s['person']:.1%}, heard the menu {s['menu']:.1%}, "
          f"US${min(cost[menu]):.0f} per 1,000 calls")
print("Everything to a person: US$", outcome_cost(tally([route(None, False, 'x')] * 10)), "per 1,000 calls")

plt.figure(figsize=(7, 3.2))
plt.plot(thresholds, cost[False], label="sure, else a person")
plt.plot(thresholds, cost[True], label="sure, else a 3-item menu")
plt.xlabel("route when the top probability is at least"); plt.ylabel("US$ per 1,000 calls"); plt.ylim(0, None)
plt.legend(); plt.show()

> **Check.** Without a menu: threshold 0.30, 90.9% routed right, US$552 per 1,000 calls. With the short menu: threshold 0.60, 96.7% routed right, US$156 per 1,000 calls, and 22.4% of the callers hear the extra menu. With a safety net, the IVR can afford to be less bold.

## 8. Change one thing: is French harder, or does it just have less data?

The English model learned from three varieties together. Give each variety its own model, with about the same number of requests.

In [ ]:
for v in VARIETIES:
    sel = [i for i in comp if by_id[i]["variety"] == v]
    p = cross_val([TEXT["base"][i] for i in sel], [by_id[i]["intent"] for i in sel], [by_id[i]["intent"] for i in sel])
    acc = np.mean([INTENTS[int(np.argmax(p[k]))] == by_id[i]["intent"] for k, i in enumerate(sel)])
    print(f"{v}: its own model, {len(sel)} requests, accuracy {acc:.1%}")

> **Check.** Each English variety alone still scores about 95% or more; French stays far lower. The gap is not only the amount of data: the French transcripts are worse (section 2).

## 9. Change one thing: Whisper small

The open answer is a whole sentence. Does a bigger recogniser help the classifier, and what does it cost in time?

In [ ]:
P_small = oof("small", comp)
top3_small = {i: [INTENTS[k] for k in np.argsort(-P_small[i])[:3]] for i in comp}
clf_small = {i: top3_small[i][0] for i in comp}
print(f"Classifier on Whisper small: {accuracy(clf_small, comp):.1%}", by_variety(clf_small, comp))
cost_small = [outcome_cost(shares(comp, t, True, P_small, top3_small), per_call=PER_CALL_SMALL) for t in thresholds]
T = thresholds[int(np.argmin(cost_small))]
s = shares(comp, T, True, P_small, top3_small)
print(f"Short-menu design on Whisper small: threshold {T:.2f}, routed right {s['right']:.1%}, heard the menu {s['menu']:.1%}, US${min(cost_small):.0f} per 1,000 calls")
for w in ("base", "small"):
    print(f"Whisper {w}: median {np.median([r['seconds'] for r in whisper[w].values()]):.2f} s of compute per request")

> **Check.** French rises to 93.0%. The short-menu design on Whisper small: threshold 0.60, US$118 per 1,000 calls. Each request takes about three times as long to transcribe.

## 10. End to end on new input

Train the chosen design (Whisper small, the classifier, route when sure, else a short menu) on all comparison requests and run it once on the requests kept apart.

In [ ]:
models = {}
for lang in ("en", "fr"):
    sel = [i for i in comp if by_id[i]["language"] == lang]
    models[lang] = IntentClassifier().fit([TEXT["small"][i] for i in sel], [by_id[i]["intent"] for i in sel])

def decide(text, language):
    """The chosen intent step: route when sure, else a short menu of the top 3 intents."""
    cls, p = models[language].proba([text])
    order = np.argsort(-p[0])
    top = [cls[k] for k in order[:3]]
    return top, float(p[0][order[0]]), ("route" if p[0][order[0]] >= T else "menu")

outcomes = []
for i in new_ids:
    top, conf, action = decide(TEXT["small"][i], by_id[i]["language"])
    outcomes.append((by_id[i]["variety"], route(top[0], action == "route", by_id[i]["intent"], top)))
for v in VARIETIES + ["all"]:
    s = tally([o for w, o in outcomes if v in ("all", w)])
    print(f"{v:6s} routed right {s['right']:.1%}  wrong {s['wrong']:.1%}  person {s['person']:.1%}  heard the menu {s['menu']:.1%}")

> **Check.** Routed right 96.9% and routed wrong 0.4% on the new input: close to the comparison, so the choice holds.

Now the whole chain, live: 12 recordings → Whisper small on your computer → the classifier → route or menu. The cell downloads Whisper small (about 480 MB) once.

In [ ]:
from faster_whisper import WhisperModel
from huggingface_hub import snapshot_download
asr = WhisperModel(snapshot_download("Systran/faster-whisper-small", revision="536b0662742c02347bc0e980a01041f333bce120"),
                   device="cpu", compute_type="int8", cpu_threads=4)
LIVE = ["en-US-151", "en-US-347", "en-US-081", "en-GB-316", "en-GB-363", "en-GB-259", "en-AU-553", "en-AU-136", "en-AU-178", "fr-FR-256", "fr-FR-219", "fr-FR-473"]
for cid in LIVE:
    c = by_id[cid]
    x, sr = sf.read(f"data/audio/{cid}.wav", dtype="float32")
    x = signal.resample_poly(x, 2, 1).astype(np.float32)        # 8 kHz -> 16 kHz, as the project does
    segments, _ = asr.transcribe(x, language=c["language"], beam_size=1, condition_on_previous_text=False)
    text = " ".join(s.text.strip() for s in segments)
    top, conf, action = decide(text, c["language"])
    said = top[0] if action == "route" else "menu: " + ", ".join(top)
    print(f"{cid} [{c['intent']}] {text!r} -> {said} ({conf:.2f}){'' if c['intent'] in top[:1 if action == 'route' else 3] else '  <- miss'}")

> **Check.** Most requests are routed right or get a menu that holds the right intent. Your Whisper text can differ a little from the recorded one.

> **Warning.** These requests were said by people who knew the intent beforehand, and cross-validation cannot keep one person's requests together (the data has no speaker IDs). Real callers will be messier: test on your own calls before you trust these numbers.

## 11. Your turn

> **Your turn.** (a) Keyword rules on the **Whisper small** transcripts (`TEXT["small"]`), French requests of the comparison set only: put the accuracy, rounded to 3 decimals, in `task_rules_fr`.

In [ ]:
rules_small = {i: find_intent(TEXT["small"][i], by_id[i]["language"]) for i in comp}
fr = [i for i in comp if by_id[i]["variety"] == "fr-FR"]
task_rules_fr = round(accuracy(rules_small, fr), 3)
print(task_rules_fr)

In [ ]:
expect('task_rules_fr', task_rules_fr, 0.555, tolerance=0.002)

> **Your turn.** (b) The cost per 1,000 calls of the menu design at threshold 0.45, on the comparison set (`shares`, `outcome_cost`, `PER_CALL`). Round to 1 decimal: `task_cost`.

In [ ]:
task_cost = round(outcome_cost(shares(comp, 0.45, True), per_call=PER_CALL), 1)
print(task_cost)

In [ ]:
expect('task_cost', task_cost, 173.1, tolerance=0.002)

> **Your turn.** (c) On the new input (`outcomes` of section 10), which variety has the lowest share routed right? Put its name (for example `"en-US"`) in `task_worst`.

In [ ]:
worst = {}
for v in VARIETIES:
    worst[v] = tally([o for w, o in outcomes if w == v])["right"]
task_worst = min(worst, key=worst.get)
print(task_worst, worst)

In [ ]:
expect_hash('task_worst', task_worst, '771b70f8605dec91')

## Next

You ran an IVR's intent step on real spoken requests, compared three methods with one measure, chose a design with its cost, and found the group it serves worst. The [case study page](https://learning.nextia-ai.com/courses/speech/m10/an-ivr-for-spoken-intents/) has the recommendation, the limits, responsible use and how to make this a portfolio piece.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [The final assignment](https://learning.nextia-ai.com/courses/speech/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m10/an-ivr-for-spoken-intents/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The spoken requests are from MInDS-14 (PolyAI, CC BY 4.0): real people who recorded requests to a bank's phone line for the dataset. The transcripts in the data were made by a speech recogniser, not by a person.